# 📱 App Store Insides 🔍

Welcome to the **App Store Insides** evaluation notebook! 🎉

This interactive notebook demonstrates the workflow for analyzing app store feedback with AI-powered insights.

## 🚀 What You'll Do

| Step | Description | Icon |
|------|-------------|------|
| **1** | Fetch reviews from Apple App Store & Google Play Store | 🍎 🤖 |
| **2** | Visualize rating distributions with interactive charts | 📊 📈 |
| **3** | Classify feedback using OpenAI AI | 🤖 💡 |
| **4** | Analyze category distributions and patterns | 🔍 📉 |
| **5** | Generate actionable insights | 💡 ✨ |

Let's get started! 👇

## 1️⃣ Setup and Configuration

### 🔧 Import Libraries & Configure Display Settings

First, we'll import all the necessary tools and configure how data is displayed in the notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add src to path for local development
SRC_PATH = Path.cwd() / "src"
sys.path.insert(0, str(SRC_PATH))

from app_store_insides import (
    AppStoreFetcher,
    FeedbackClassifier,
    GooglePlayFetcher,
    RatingVisualizer,
)
from app_store_insides.data_fetcher import combine_reviews

print("✅ Imports successful!")

## 2️⃣ Fetch Reviews from App Stores

### 🎯 Configure Your Apps

Set up the app IDs for the applications you want to analyze. You can analyze apps from both Apple App Store and Google Play Store.

**How to find App IDs:**
- 🍎 **Apple:** `apps.apple.com/.../id[APP_ID]` - use the numeric ID
- 🤖 **Google:** `play.google.com/store/apps/details?id=[PACKAGE_NAME]` - use the package name

In [ ]:
# Configuration
APPLE_APP_ID = 348712880  # Example: Amazon
GOOGLE_APP_ID = "com.amazon.mShop.android.shopping"  # Example: Amazon
COUNTRY = "de"
LANGUAGE = "de"

COUNT_OF_COMMENTS = 500

print(f"🍎 Fetching Apple App Store reviews for app ID: {APPLE_APP_ID} [Link https://apps.apple.com/{COUNTRY}/app/eurowings/id{APPLE_APP_ID}]")
print(f"🤖 Fetching Google Play Store reviews for app ID: {GOOGLE_APP_ID} [Link https://play.google.com/store/apps/details?id={GOOGLE_APP_ID}&hl={LANGUAGE}&gl={COUNTRY}]")

### 🍎 Fetch Apple App Store Reviews

Now we'll fetch reviews from the Apple App Store. This may take a moment depending on how many reviews you're fetching.

In [ ]:
# Fetch Apple App Store reviews
apple_fetcher = AppStoreFetcher(app_id=APPLE_APP_ID, country=COUNTRY)
apple_reviews = apple_fetcher.fetch_reviews(limit=COUNT_OF_COMMENTS)  # Limit for faster testing

print(f"✅ Fetched {len(apple_reviews)} Apple App Store reviews")
apple_reviews.head()

### 🤖 Fetch Google Play Store Reviews

Next, we'll fetch reviews from the Google Play Store.

In [ ]:
# Fetch Google Play Store reviews
google_fetcher = GooglePlayFetcher(app_id=GOOGLE_APP_ID, country=COUNTRY, lang=LANGUAGE)
google_reviews = google_fetcher.fetch_reviews(limit=COUNT_OF_COMMENTS)

print(f"✅ Fetched {len(google_reviews)} Google Play Store reviews")
google_reviews.head()

### 🔄 Combine All Reviews

Let's merge the reviews from both platforms into a single dataset for unified analysis.

In [ ]:
# Make apple_reviews and google_reviews optional with empty DataFrames as default
if 'apple_reviews' not in dir() or apple_reviews is None:
    apple_reviews = pd.DataFrame(columns=['id', 'rating', 'review', 'date', 'platform'])
if 'google_reviews' not in dir() or google_reviews is None:
    google_reviews = pd.DataFrame(columns=['id', 'rating', 'review', 'date', 'platform'])

# Combine all reviews
all_reviews = combine_reviews(apple_reviews, google_reviews)

print(f"📊 Total reviews: {len(all_reviews)}")
print(f"   - Apple: {len(apple_reviews)}")
print(f"   - Google: {len(google_reviews)}")
print(f"\n🔢 Platform distribution:")
print(all_reviews['platform'].value_counts())

all_reviews.head()


## 3️⃣ Visualize Rating Distribution

### 📊 Create Interactive Charts

Time to visualize the data! We'll create beautiful, interactive charts to understand rating patterns across platforms.

In [ ]:
visualizer = RatingVisualizer()
fig_all = visualizer.plot_rating_distribution(
    all_reviews,
    title=f"App Store - Rating Distribution (Avg: {apple_reviews['rating'].mean():.2f})"
)
fig_all.show()

### 🍎 vs 🤖 Platform Comparison

Compare rating distributions between Apple App Store and Google Play Store to identify platform-specific trends.

In [ ]:
visualizer = RatingVisualizer()

# Rating distribution by platform
print("🍎 Apple App Store Ratings:")
fig_apple = visualizer.plot_rating_distribution(
    apple_reviews,
    title=f"Apple App Store - Rating Distribution (Avg: {apple_reviews['rating'].mean():.2f})"
)
fig_apple.show()

print("\n🤖 Google Play Store Ratings:")
fig_google = visualizer.plot_rating_distribution(
    google_reviews,
    title=f"Google Play Store - Rating Distribution (Avg: {google_reviews['rating'].mean():.2f})"
)
fig_google.show()

### 📉 Rating Trends Over Time

Discover how ratings have evolved over time with a rolling average trend line.

In [ ]:
# Rating trend over time
fig_trend = visualizer.plot_rating_over_time(
    all_reviews,
    rolling_window=20
)
fig_trend.show()

## 4️⃣ Classify Feedback with OpenAI

### 🤖 AI-Powered Review Classification

Now comes the exciting part! We'll use OpenAI's AI to automatically categorize each review into meaningful categories.

**⚠️ Important:** Make sure you have:
- Created a `.env` file in the project root
- Added your `OPENAI_API_KEY` to the `.env` file

💡 **Tip:** Classification uses the OpenAI API and incurs small costs (~$0.10-0.30 per 1000 reviews with gpt-4o-mini)

### 📋 Define Classification Categories

**✏️ Customize Your Categories Here!**

This is where you define what categories the AI should use. Feel free to:
- ➕ Add new categories
- ✏️ Modify descriptions to match your needs
- ❌ Remove categories you don't need
- 🔄 Rename categories

The AI will use these definitions to classify your reviews.

In [ ]:
# Define your classification categories and descriptions
# You can modify these categories and descriptions to match your specific needs

CLASSIFICATION_CATEGORIES = {
    "CRITICAL_BUG": "User reports a malfunction that blocks usage or causes major issues (e.g. data loss, crashes, login failures)",
    "MINOR_BUG": "User reports a small or cosmetic issue that doesn't break core functionality",
    "MISSING_CORE_FEATURE": "User requests a feature that is standard in similar apps or essential for usability",
    "MISSING_NICE_TO_HAVE": "User requests an optional or convenience feature (e.g. dark mode, customization)",
    "PAINFUL_UX": "User describes friction, confusion, or inefficiency in the interface or user flow",
    "PERFORMANCE_BOTTLENECK": "User experiences slowness, unresponsiveness, or long loading times",
    "NEGATIVE_EMOTION": "User expresses frustration, anger, disappointment, or distrust, regardless of technical cause",
    "POSITIVE_EMOTION": "User expresses delight, appreciation, or love for the app or a specific feature",
    "BRAND_COMPARISON": "User compares the app negatively or positively to a competitor (valuable for benchmarking)",
    "LACK_OF_PERSONALIZATION": "User complains about having to re-enter data or inability to save preferences/settings",
    "UNMET_EXPECTATION": "User expected something based on marketing, industry standards, or previous experience but it wasn't delivered",
    "UPGRADE_VERSION_REGRESSION": "User says a recent update made things worse or removed features",
    "SUPPORT_REQUEST": "User asks for help, clarification, or contact info",
    "SECURITY_PRIVACY_CONCERN": "User raises concerns about data privacy, permissions, or trust",
    "OTHER": "Feedback that doesn't clearly fit into any category"
}


### 🎯 Initialize the AI Classifier

Setting up the classifier with your custom categories...

In [ ]:
# Initialize classifier with custom categories
classifier = FeedbackClassifier()

# Override the default categories with your custom ones
classifier.CATEGORIES = list(CLASSIFICATION_CATEGORIES.keys())
classifier.CATEGORY_DESCRIPTIONS = CLASSIFICATION_CATEGORIES

print(f"✅ Classifier initialized with {len(classifier.CATEGORIES)} categories")

### 🧪 Test Classification on a Single Review

Let's test the classifier on one review first to see how it works!

In [ ]:
# Test classification on a single review
sample_review = all_reviews.iloc[0]
result = classifier.classify_feedback(
    sample_review['review'],
    sample_review['rating']
)

print("📝 Sample Classification:")
print(f"\nReview: {sample_review['review'][:200]}...")
print(f"Rating: {sample_review['rating']}/5")
print(f"\nCategory: {result['category']}")
print(f"Explanation: {result['explanation']}")

### 🚀 Classify All Reviews (Batch Processing)

Now let's classify all the reviews! This will take a few minutes depending on how many reviews you have.

⏱️ **Estimated time:** ~5-10 seconds per review with gpt-4o-mini

In [ ]:
# Classify all reviews (this may take a while depending on the number of reviews)
# For testing, you can limit to first n reviews
sample_size = 50  # Adjust this number
reviews_to_classify = all_reviews.head(sample_size)

print(f"🔄 Classifying {len(reviews_to_classify)} reviews...\n")
classified_reviews = classifier.classify_batch(reviews_to_classify)

print(f"\n✅ Classification complete!")
classified_reviews.head(10)

## 5️⃣ Analyze Category Distribution

### 📊 Understanding Your Feedback Categories

Let's analyze how the reviews are distributed across different categories and what that tells us about user sentiment.

In [ ]:
# Get category distribution
category_dist = classifier.get_category_distribution(classified_reviews)

print("📊 Feedback Category Distribution:\n")
print(category_dist.to_string(index=False))

### 🥧 Category Distribution - Pie Chart

Visual representation of category proportions.

In [ ]:
# Visualize category distribution - Pie Chart
fig_pie = visualizer.plot_category_distribution(classified_reviews)
fig_pie.show()

### 📋 Category Distribution Table

Here's a breakdown of how many reviews fall into each category.

In [ ]:
# Visualize category distribution - Bar Chart
fig_bar = visualizer.plot_category_bar_chart(classified_reviews)
fig_bar.show()

### ⭐ Average Rating by Category

Which categories have the highest and lowest ratings? This helps identify problem areas!

In [ ]:
# Average rating by category
fig_rating_by_cat = visualizer.plot_rating_by_category(classified_reviews)
fig_rating_by_cat.show()

## 6️⃣ Explore Specific Categories

### 🔍 Deep Dive into Feedback Categories

### 📑 List All Available Categories

See which categories were found in your reviews.

In [ ]:
# Show all categories as a DataFrame
category_summary = pd.DataFrame([
    {'Category': cat, 'Count': len(classified_reviews[classified_reviews['category'] == cat])}
    for cat in classified_reviews['category'].unique()
]).sort_values('Count', ascending=False).reset_index(drop=True)

category_summary


### 🎯 Explore a Specific Category

Pick a category and read through actual reviews to understand the issues or praise in detail.

💡 **Tip:** Change `CATEGORY_TO_EXPLORE` to any category you want to investigate!

In [ ]:
# Explore a specific category (change as needed)
CATEGORY_TO_EXPLORE = "MISSING_CORE_FEATURE"  # Change this to any category

category_reviews = classified_reviews[
    classified_reviews['category'] == CATEGORY_TO_EXPLORE
    ]

print(f"\n📋 {CATEGORY_TO_EXPLORE} Reviews ({len(category_reviews)} total)\n")

# Display as DataFrame with selected columns
category_reviews[['rating', 'platform', 'review', 'explanation']]


## 7️⃣ Export Results

### 💾 Save Your Analysis

In [ ]:
# Export to CSV
output_file = "classified_reviews.csv"
classified_reviews.to_csv(output_file, index=False)
print(f"✅ Exported {len(classified_reviews)} classified reviews to {output_file}")